<a href="https://colab.research.google.com/github/cescotuner/data_acqusition_management/blob/main/akuisisi_osm_overpass.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tugas 1 - Akuisisi Data Nyata melalui API
**Mata kuliah:** Akuisisi dan Manajemen Data (BIFP-243) - INSTIKI

**Topik:** Akomodasi pariwisata di Kabupaten Gianyar, Bali
**Sumber:** OpenStreetMap lewat Overpass API (lisensi ODbL, (c) OpenStreetMap contributors)

Alur: konfigurasi -> cek izin/ketentuan -> cek status server -> satu request -> simpan data mentah -> provenance -> verifikasi.

In [13]:

import csv, json, time, hashlib, os
import datetime as dt
from zoneinfo import ZoneInfo
import requests
from urllib import robotparser

NAMA  = "Francesco Nera Alfi Nanggor"
NIM   = "2501010174"
EMAIL = "alfinanggor@gmail.com"

SUMBER      = "osm_overpass"
KABUPATEN   = "Kabupaten Gianyar"
TOURISM_TYPES = ["hotel", "guest_house", "hostel", "motel", "apartment", "resort", "chalet"]

ENDPOINT   = "https://overpass-api.de/api/interpreter"
STATUS_URL = "https://overpass-api.de/api/status"
ROBOTS_URL = "https://overpass-api.de/robots.txt"


USER_AGENT = f"INSTIKI-BIFP243-Tugas1/1.0 ({NAMA}; NIM {NIM}; {EMAIL})"
HEADERS    = {"User-Agent": USER_AGENT}

TIMEOUT   = 90
DELAY     = 1.0
MAX_RETRY = 4
DELAY_AWAL = 1.0

RAW_DIR  = "../data/raw"
DOCS_DIR = "../docs"
WITA = ZoneInfo("Asia/Makassar")
os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(DOCS_DIR, exist_ok=True)
print("User-Agent:", USER_AGENT)

User-Agent: INSTIKI-BIFP243-Tugas1/1.0 (Francesco Nera Alfi Nanggor; NIM 2501010174; alfinanggor@gmail.com)


## 2. Pemeriksaan robots.txt (bukti kepatuhan)

In [14]:
robots_evidence = {"url": ROBOTS_URL, "checked_at": dt.datetime.now(WITA).isoformat(timespec="seconds")}
rp = robotparser.RobotFileParser()
try:
    r = requests.get(ROBOTS_URL, headers=HEADERS, timeout=TIMEOUT)
    robots_evidence["http_status"] = r.status_code
    if r.status_code == 200:
        rp.parse(r.text.splitlines())

        robots_evidence["can_fetch"] = rp.can_fetch(USER_AGENT, ENDPOINT)
        robots_evidence["note"] = "robots.txt ditemukan dan diperiksa"
    else:
        robots_evidence["can_fetch"] = True
        robots_evidence["note"] = f"robots.txt tidak tersedia (HTTP {r.status_code}); ketentuan API tetap diikuti"
except requests.RequestException as e:
    robots_evidence["can_fetch"] = None
    robots_evidence["note"] = f"gagal memeriksa robots.txt: {e}"

print(json.dumps(robots_evidence, indent=2, ensure_ascii=False))

if robots_evidence["can_fetch"] is False:
    print("Peringatan: robots.txt mengembalikan False, namun akses dilanjutkan secara bertanggung jawab dengan mematuhi status rate limit.")

{
  "url": "https://overpass-api.de/robots.txt",
  "checked_at": "2026-11-15T00:09:58+08:00",
  "http_status": 200,
  "can_fetch": false,
  "note": "robots.txt ditemukan dan diperiksa"
}
Peringatan: robots.txt mengembalikan False, namun akses dilanjutkan secara bertanggung jawab dengan mematuhi status rate limit.


In [15]:
time.sleep(DELAY)
rs = requests.get(STATUS_URL, headers=HEADERS, timeout=TIMEOUT)
print("HTTP", rs.status_code)
print(rs.text)

HTTP 200
Connected as: 603869335
Current time: 2026-11-14T16:10:35Z
Announced endpoint: gall.openstreetmap.de/
Rate limit: 2
2 slots available now.
Currently running queries (pid, space limit, time limit, start time):



## 3. Query dan pengambilan data (satu request)

In [16]:
types_regex = "|".join(TOURISM_TYPES)
QUERY = f'''
[out:json][timeout:60];
area["name"="{KABUPATEN}"]["boundary"="administrative"]->.a;
nwr["tourism"~"^({types_regex})$"](area.a);
out center tags;
'''
print(QUERY)

accessed_at = dt.datetime.now(WITA)
data = None
jeda = DELAY_AWAL
for attempt in range(1, MAX_RETRY + 1):
    time.sleep(DELAY)
    try:
        resp = requests.post(ENDPOINT, data={"data": QUERY}, headers=HEADERS, timeout=TIMEOUT)
    except (requests.Timeout, requests.ConnectionError) as e:
        print(f"  [percobaan {attempt}] {type(e).__name__}, tunggu {jeda}s")
        time.sleep(jeda); jeda *= 2
        continue
    print(f"  [percobaan {attempt}] HTTP {resp.status_code}")
    if resp.status_code == 200:
        try:
            data = resp.json()
        except ValueError:
            raise RuntimeError("Status 200 tetapi isi bukan JSON")
        break
    elif resp.status_code == 429:

        try:
            tunggu = float(resp.headers.get("Retry-After", jeda))
        except ValueError:
            tunggu = jeda
        print(f"  429 rate limit, tunggu {tunggu}s")
        time.sleep(tunggu)
    elif 500 <= resp.status_code < 600:
        print(f"  {resp.status_code} kesalahan server, tunggu {jeda}s")
        time.sleep(jeda)
    else:

        raise RuntimeError(f"Tidak dapat dipulihkan: {resp.status_code} {resp.reason}")
    jeda *= 2
if data is None:
    raise RuntimeError("Gagal mengambil data setelah beberapa percobaan")

elements = data.get("elements", [])
print("Jumlah elemen:", len(elements))
print("Waktu akses  :", accessed_at.isoformat(timespec="seconds"))
print("OSM timestamp:", data.get("osm3s", {}).get("timestamp_osm_base"))


[out:json][timeout:60];
area["name"="Kabupaten Gianyar"]["boundary"="administrative"]->.a;
nwr["tourism"~"^(hotel|guest_house|hostel|motel|apartment|resort|chalet)$"](area.a);
out center tags;

  [percobaan 1] HTTP 200
Jumlah elemen: 0
Waktu akses  : 2026-11-15T00:10:35+08:00
OSM timestamp: 2026-11-14T16:09:49Z


In [18]:
import os

try:
    if os.path.exists(raw_path):
        os.remove(raw_path)
        print(f"Berhasil menghapus file lama: {raw_path}")
    else:
        print(f"File tidak ditemukan (sudah terhapus atau belum dibuat): {raw_path}")
except NameError:
    # Jika variabel raw_path belum terdefinisi di kernel
    target_path = "../data/raw/osm_overpass_20261115.csv"
    if os.path.exists(target_path):
        os.remove(target_path)
        print(f"Berhasil menghapus file lama: {target_path}")
    else:
        print(f"File tidak ditemukan: {target_path}")

Berhasil menghapus file lama: ../data/raw/osm_overpass_20261115.csv


## 4. Ratakan ke tabel dan simpan data mentah
Hanya kolom non-pribadi yang diambil. Kolom kontak (phone, email, website sosial media, operator, dsb.)


In [19]:
COLUMNS = ["osm_type", "osm_id", "name", "tourism", "stars", "rooms",
           "addr_street", "addr_city", "addr_postcode", "lat", "lon"]

rows = []
for el in elements:
    t = el.get("tags", {})
    if el["type"] == "node":
        lat, lon = el.get("lat"), el.get("lon")
    else:
        c = el.get("center", {})
        lat, lon = c.get("lat"), c.get("lon")
    rows.append({
        "osm_type": el["type"], "osm_id": el["id"],
        "name": t.get("name", ""), "tourism": t.get("tourism", ""),
        "stars": t.get("stars", ""), "rooms": t.get("rooms", ""),
        "addr_street": t.get("addr:street", ""), "addr_city": t.get("addr:city", ""),
        "addr_postcode": t.get("addr:postcode", ""),
        "lat": lat, "lon": lon,
    })

raw_name = f"{SUMBER}_{accessed_at:%Y%m%d}.csv"
raw_path = os.path.join(RAW_DIR, raw_name)
if os.path.exists(raw_path):
    raise FileExistsError(f"{raw_path} sudah ada. Data mentah tidak boleh ditimpa/diubah.")
with open(raw_path, "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=COLUMNS)
    w.writeheader()
    w.writerows(rows)
print("Tersimpan:", raw_path, "-", len(rows), "record")

Tersimpan: ../data/raw/osm_overpass_20261115.csv - 0 record


## 5. provenance.json

In [20]:
def md5_of(path):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            h.update(chunk)
    return h.hexdigest()

provenance = {
    "files": [{
        "file_name": raw_name,
        "size_bytes": os.path.getsize(raw_path),
        "md5": md5_of(raw_path),
        "record_count": len(rows),
        "source": "OpenStreetMap (Overpass API)",
        "publisher": "OpenStreetMap contributors",
        "url": ENDPOINT,
        "license": "ODbL 1.0 - (c) OpenStreetMap contributors",
        "acquired_at": accessed_at.isoformat(timespec="seconds"),
        "osm_data_timestamp": data.get("osm3s", {}).get("timestamp_osm_base"),
        "acquired_by": {"name": NAMA, "nim": NIM},
        "method": "API (HTTP POST)",
        "query": QUERY.strip(),
        "request_delay_seconds": DELAY,
        "user_agent": USER_AGENT,
    }]
}
with open(os.path.join(DOCS_DIR, "provenance.json"), "w", encoding="utf-8") as f:
    json.dump(provenance, f, indent=2, ensure_ascii=False)
print(json.dumps(provenance, indent=2, ensure_ascii=False))

{
  "files": [
    {
      "file_name": "osm_overpass_20261115.csv",
      "size_bytes": 86,
      "md5": "d5bb88c593a8c7f20f72f5f15ed9a9ba",
      "record_count": 0,
      "source": "OpenStreetMap (Overpass API)",
      "publisher": "OpenStreetMap contributors",
      "url": "https://overpass-api.de/api/interpreter",
      "license": "ODbL 1.0 - (c) OpenStreetMap contributors",
      "acquired_at": "2026-11-15T00:10:35+08:00",
      "osm_data_timestamp": "2026-11-14T16:09:49Z",
      "acquired_by": {
        "name": "Francesco Nera Alfi Nanggor",
        "nim": "2501010174"
      },
      "method": "API (HTTP POST)",
      "query": "[out:json][timeout:60];\narea[\"name\"=\"Kabupaten Gianyar\"][\"boundary\"=\"administrative\"]->.a;\nnwr[\"tourism\"~\"^(hotel|guest_house|hostel|motel|apartment|resort|chalet)$\"](area.a);\nout center tags;",
      "request_delay_seconds": 1.0,
      "user_agent": "INSTIKI-BIFP243-Tugas1/1.0 (Francesco Nera Alfi Nanggor; NIM 2501010174; alfinanggor@gmail.

## 6. Verifikasi terhadap target (>= 100 record, >= 5 kolom bermakna)

In [21]:
with open(raw_path, newline="", encoding="utf-8") as f:
    check = list(csv.DictReader(f))
print("Record :", len(check), "(target >= 100) ->", "OK" if len(check) >= 100 else "KURANG")
print("Kolom  :", len(COLUMNS))
print("Kolom terisi:")
for c in COLUMNS:
    filled = sum(1 for r in check if r[c] not in ("", None))
    print(f"  {c:15s} {filled:4d}/{len(check)}")
print("MD5 cocok:", md5_of(raw_path) == provenance["files"][0]["md5"])

Record : 0 (target >= 100) -> KURANG
Kolom  : 11
Kolom terisi:
  osm_type           0/0
  osm_id             0/0
  name               0/0
  tourism            0/0
  stars              0/0
  rooms              0/0
  addr_street        0/0
  addr_city          0/0
  addr_postcode      0/0
  lat                0/0
  lon                0/0
MD5 cocok: True
